In [1]:
%load_ext autoreload
%autoreload 2

# Entrenamiento del modelo

Con el dataset listo, procederemos a configurar y entrenar nuestra red neuronal. Este proceso se divide en la carga de datos, la definición de la arquitectura y la ejecución del ciclo de entrenamiento supervisado.

### Carga del dataset

Primero, importamos los datos preparados. En este paso, transformamos el archivo `.data` en un objeto manejable por el framework para el entrenamiento.

In [ ]:
from preprocessing.dataset import load_dataset

# Cargamos el dataset que balanceamos en pasos anteriores
filename = "dataset_ejemplo_uc.data"
data = load_dataset(filename)

Dataset dataset_ejemplo_uc.data cargado con 1000 muestras.


### Configuración de la arquitectura

Definiremos los hiperparámetros del **CPMPTransformer**. Es fundamental que las dimensiones de entrada (`H_dim`, `C_dim`, `X_dim`) coincidan con el formato del **Input Adapter** utilizado durante la generación de los datos.

In [ ]:
from models.actions.cpmp_transformer_V2 import CPMPTransformer

# Parámetros de la arquitectura
model = CPMPTransformer(
    H_dim=12,                # Dimensión de altura (máx stacks)
    C_dim=3,                 # Features por contenedor
    X_dim=3,                 # Features globales del stack
    d_model=32,              # Dimensión interna del Transformer
    nhead=4,                 # Número de cabezales de atención
    num_layers=4,            # Número de capas del encoder
    ff_dim_multiplier=2,     # Multiplicador de la capa Feed-Forward
    dropout=0.3              # Regularización para evitar overfitting
)

### Ejecución del entrenamiento supervisado (SL)

Utilizaremos la función `sl_train`. Esta se encarga de dividir los datos en conjuntos de entrenamiento y validación, gestionar el optimizador y aplicar **Early Stopping** mediante el parámetro `patience`.

In [9]:
from training.training import sl_train, Accuracy, CrossEntropyLoss

# Configuración del ciclo de entrenamiento
epochs = 100
train_size = 800
test_size = 200
batch_size = 8
learning_rate = 1e-4

# Definimos la función de pérdida y las métricas de evaluación
loss_function = CrossEntropyLoss()
metrics = [Accuracy()]

# Iniciamos el proceso de entrenamiento
model = sl_train(
    model, epochs, data, train_size, test_size, batch_size, 
    learning_rate, weight_decay=1e-4, 
    loss_functions=[loss_function], 
    patience=10, 
    metrics=[metrics], 
    seed=42
)

ℹ️ Usando dispositivo: cpu

Epoch 1/100
    Train CrossEntropy: 2.6846
    Val CrossEntropy: 2.6434
 | Accuracy: 18.00%
Epoch 2/100
    Train CrossEntropy: 2.6796
    Val CrossEntropy: 2.6387
 | Accuracy: 24.00%
Epoch 3/100
    Train CrossEntropy: 2.6724
    Val CrossEntropy: 2.6331
 | Accuracy: 24.50%
Epoch 4/100
    Train CrossEntropy: 2.6633
    Val CrossEntropy: 2.6207
 | Accuracy: 24.00%
Epoch 5/100
    Train CrossEntropy: 2.6493
    Val CrossEntropy: 2.6099
 | Accuracy: 21.50%
Epoch 6/100
    Train CrossEntropy: 2.6394
    Val CrossEntropy: 2.5899
 | Accuracy: 22.00%
Epoch 7/100
    Train CrossEntropy: 2.6405
    Val CrossEntropy: 2.5723
 | Accuracy: 21.00%
Epoch 8/100
    Train CrossEntropy: 2.6252
    Val CrossEntropy: 2.5701
 | Accuracy: 20.50%
Epoch 9/100
    Train CrossEntropy: 2.6154
    Val CrossEntropy: 2.5690
 | Accuracy: 19.00%
Epoch 10/100
    Train CrossEntropy: 2.6132
    Val CrossEntropy: 2.5545
 | Accuracy: 20.00%
Epoch 11/100
    Train CrossEntropy: 2.5999
    Val

### Persistencia del modelo

Una vez finalizado el entrenamiento, guardaremos el modelo. Esto nos permitirá cargarlo posteriormente en la parte de **Validación** para resolver instancias de benchmark.

In [10]:
from training.training import save_model

# Guardamos los pesos y la configuración del modelo bajo un nombre identificativo
save_model(model, "ejemplo_transformer_sl")

✅ Modelo guardado en /home/oscar/Escritorio/CPMP-Framework/models/ejemplo_transformer_sl.pth
